# 1 Bias variance Trade-off of Ridged regression

We have a linear model of the form $y = X\beta + \epsilon$

Now we ridge estimator is the following:

$$
\hat{\beta}_{\text{ridge}} = (X^T X + \lambda I)^{-1} X^T y
$$


Now we calcualte the squared error for this point
$$ (\hat{y_{0}}-y_{0})^2 = (x_{0}^T\hat{\beta}_{\text{ridge}} - x_{0}^T\beta) ^2 + ϵ_{0}$$

$$
\mathbb{E}[\hat{\beta}_{\text{ridge}}] = (X^T X + \lambda I)^{-1} X^T X \beta
$$

And now we can calculate the bias of the coefficent estimate

$$ Bias[\hat{\beta}_{\text{ridge}}] = \mathbb{E}[\hat{\beta}_{\text{ridge}}] - \hat{\beta}_{\text{ridge}} = ((X^T X + \lambda I)^{-1} X^T X - I)\beta
$$

Now we will calculate the variance of the $\hat{\beta}_{\text{ridge}}$

$$
\mathrm{Var}(\hat{\beta}_{\text{ridge}}) = \sigma^2 (X^T X + \lambda I)^{-1} X^T X (X^T X + \lambda I)^{-1}
$$

Now as $\lambda$ increases we get that the term $(X^T X + \lambda I) $ becomes more diagonal and therefore $(X^T X + \lambda I)^{-1} $ becomes smaller. Therefore when we increase $\lambda$ the variance will become 0 however the magnitude of the bias will increase. On the other hand if $\lambda$ will decrease the bias will be closer to 0 and the variance will become bigger.


Now we will find the $\lambda$ that gives the minimum value. we will calculate the MSE of the coefficients

$$
\sum_{j=1}^{p} \mathbb{E}\left(\frac{1}{1+\lambda} \hat{\beta}_j - \beta_j \right)^2 = \left(\frac{1}{1+\lambda}\right)^2 \sum_{j=1}^{p} \mathbb{E} \left(\hat{\beta}_j - \beta_j \right)^2 + \left(\frac{\lambda}{1+\lambda}\right)^2 \sum_{j=1}^{p} \beta_j^2
$$

$$
= p \sigma^2 \left(\frac{1}{1+\lambda}\right)^2 + \left(\frac{\lambda}{1+\lambda}\right)^2 \sum_{j=1}^{p} \beta_j^2
$$

After taking the derivative and making everythign equal to 0, we get that the $\alpha$ with the smallest MSE is the following

$$
\lambda^* = \frac{p \sigma^2}{\sum_{j=1}^{p} \beta_j^2}
$$



# 2 Beyond quadratic loss

The integraldepends on the value of $\alpha\ $ to calculate the optimal value. This optimal value is the minimizer of the $L_{1}$ loss. This happens then $α$ is equal to the median of y cause then the probability mass function on both the left and the right are equal therefore we would have the smallest loss the

Financial interpretation: The difference in the predicted and actual price of a stock at time T given the price at time t is the smallest when the prediction is the median value of the stock price across all possible periods of time available

If we had x in multiple dimensions, when y would need to be calculated based on the point where it has a geometric median across all dimensions, which is possible and it will not change the actual definition.

# Part 3 Feature Engineering on hedge fund dataset

# Part 3: Hedge Fund Factor Analysis (Demo)

The analysis logic lives in `src/hedge_fund_factor_modeling/`. This notebook demonstrates the pipeline using the sample dataset.

Run from the repo root:
```bash
pip install -e .
python -m hedge_fund_factor_modeling.cli train --model all
```

In [ ]:
import os
from pathlib import Path

# Run from repo root so config paths resolve correctly
repo_root = Path.cwd()
if (repo_root / 'configs').exists():
    os.chdir(repo_root)
elif (repo_root.parent / 'configs').exists():
    os.chdir(repo_root.parent)

import pandas as pd
from hedge_fund_factor_modeling.data import load_config
from hedge_fund_factor_modeling.cli import prepare_data

config = load_config('configs/default.yaml')
merged = prepare_data('configs/default.yaml')
merged.head()


In [ ]:
split = time_series_split(
    merged,
    factor_cols=config["factors"]["factor_cols"],
    target_col=config["factors"]["target_col"],
    train_end=config["split"]["train_end"],
    test_start=config["split"]["test_start"],
)
print(f"Train: {split.train_index.min().date()} → {split.train_index.max().date()}")
print(f"Test:  {split.test_index.min().date()} → {split.test_index.max().date()}")

In [ ]:
ridge = train_model("ridge", split, config)
ols = train_model("ols", split, config)

for name, model in [("OLS", ols), ("Ridge", ridge)]:
    metrics = evaluate_split(model, split, name.lower(), predict_model)
    print(name, metrics)

In [ ]:
rolling = rolling_factor_betas(
    merged,
    factor_cols=config["factors"]["factor_cols"],
    target_col=config["factors"]["target_col"],
    window=config["rolling"]["window_months"],
    min_periods=config["rolling"]["min_periods"],
    alpha=config["regression"]["ridge_alpha"],
)
rolling.tail()

In [ ]:
performance = pd.read_csv("../results/model_performance.csv")
performance

In [ ]:
from IPython.display import Image, display
for img in [
    "../assets/factor_exposure_heatmap.png",
    "../assets/rolling_beta_chart.png",
    "../assets/predicted_vs_actual_returns.png",
]:
    display(Image(filename=img))